# NBA Points Modeling Research (v1)

Context:
- Goal: build a game-to-game points modeling workflow, graduating from exploratory notebook → pipeline scripts.
- Focus in this notebook: data sanity, target construction, feature diagnostics, baseline model, edge calibration.
- Scope: research only (no production assumptions yet).
- Season hard-coded to 2025-26 throughout; change `SEASON` below to explore others.

Data source: `s3://nba-betting-mt/data/03_intermediate/player_props_with_actuals_{season}.csv`
- One row per player-game (already points-only, aggregated across bookmakers)
- Key columns: `points_line`, `points_over_odds`, `points_under_odds`, `PTS`, `MIN`, `team_spread`, `scorer_type`, `is_home`

In [1]:
from pathlib import Path
import subprocess
import sys

import boto3
import numpy as np
import pandas as pd
from io import StringIO

repo_root = Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)
module_root = repo_root / "src" / "nba_points_modeling"
research_root = module_root / "research"

sys.path.insert(0, str(repo_root / "src"))

# Hard-coded season — change here to explore other seasons
SEASON = "2025-26"

S3_BUCKET_INTERMEDIATE = "nba-betting-mt"
S3_KEY_INTERMEDIATE = f"data/03_intermediate/player_props_with_actuals_{SEASON}.csv"

print(f"repo_root={repo_root}")
print(f"module_root={module_root}")
print(f"season={SEASON}")

repo_root=/Users/thomasmyles/dev/betting
module_root=/Users/thomasmyles/dev/betting/src/nba_points_modeling
season=2025-26


---
## 1. Load Data

In [2]:
import importlib.util, sys as _sys

_joiner_path = repo_root / "scripts" / "join_nba_points_props_actuals_charts_gamelines.py"
_spec = importlib.util.spec_from_file_location("joiner", _joiner_path)
joiner = importlib.util.module_from_spec(_spec)
_sys.modules["joiner"] = joiner
_spec.loader.exec_module(joiner)

# Loads props + game logs + shot charts + game lines fresh from S3 — no stale intermediate
# rim_scorer_pct=40: players with ≥40% of pts from 0-6 ft classified as rim attackers
df_raw = joiner.join_all_data(SEASON, rim_scorer_pct=40)
print(f"\nrows={len(df_raw):,}  cols={df_raw.shape[1]}")
print(f"date range: {df_raw['game_date'].min()} → {df_raw['game_date'].max()}")
df_raw.head(3)


LOADING ALL DATA FOR 2025-26

📊 Loading props from s3://the-odds-api-mt/nba/historical_player_props/2025-26/...


✅ Loaded 751,294 prop rows from 200 files
   Dates: 2023-11-21 to 2026-05-18
   Unique players: 433
   Markets: ['player_assists', 'player_blocks', 'player_double_double', 'player_points', 'player_points_rebounds_assists', 'player_rebounds', 'player_steals', 'player_threes', 'player_triple_double']

🏀 Loading game logs from s3://nba-api-mt/player_game_logs/2025-26/...


✅ Loaded 18,608 player-game rows from 195 files
   Dates: 2025-10-21 to 2026-02-23
   Unique players: 538

🎯 Loading shot charts from s3://nba-api-mt/player_shot_charts/2025-26/...


  ⚠️  Error loading player_shot_charts/2025-26/Tamar_Bates_1642926.csv: 'SHOT_DISTANCE'


  ⚠️  Error loading player_shot_charts/2025-26/Terry_Rozier_1626179.csv: 'SHOT_DISTANCE'
  ⚠️  Error loading player_shot_charts/2025-26/Thomas_Sorber_1642850.csv: 'SHOT_DISTANCE'


...

  ⚠️  Error loading player_shot_charts/2025-26/Vladislav_Goldin_1642884.csv: 'SHOT_DISTANCE'


✅ Loaded 499 player shot chart aggregations
   Average rim FG%: 58.3%
   Average rim points per player: 111.9

📈 Loading game lines from s3://the-odds-api-mt/nba/historical_game_lines/2025-26/...


/Users/thomasmyles/dev/betting/scripts/join_nba_points_props_actuals_charts_gamelines.py:254: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_lines = pd.concat(all_lines, ignore_index=True)


✅ Loaded 1,307 unique games with consensus lines
   Dates: 2025-10-21 to 2026-05-17
   Unique games: 1,307

🎯 Calculating scorer type classification (0-6 feet vs total points, rim_scorer_pct=40%)...
   Players with game logs: 537
✅ Scorer type classification complete:
   Rim Attacker (≥40%) from 0-6 ft: 128 (25.7%)
   Perimeter (<40%) from 0-6 ft: 371 (74.3%)
   Average pts from 0-6 ft: 29.6%

JOINING DATA

Starting with game logs as base...
Filtering to player_points market only...
   207,094 player_points prop rows
Aggregating props by player/date...
Left joining player_points props to game logs...
✅ Player points props joined
Left joining shot charts with scorer type classification...
✅ Shot charts joined (includes pts_0_6_pct and scorer_type)
Joining game lines...
✅ Game lines joined

FINAL MERGED DATASET
Total rows: 18,608
Total columns: 51
Date range: 2025-10-21 to 2026-02-23

📊 Scorer Type Distribution:
   Per Player (537 unique players):
      Perimeter (<40%): 360 players (67.

,PLAYER_ID,PLAYER_NAME,PLAYER_NAME_NORMALIZED,TEAM_ID,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,...,total_pts_season,pts_0_6_pct,scorer_type,rim_scorer_threshold,team_spread,team_spread_odds,team_moneyline,opponent_team,is_home,is_favorite
0,1629029,Luka Dončić,Luka Doncic,1610612747,Los Angeles Lakers,22500002,2025-10-21,LAL vs. GSW,L,40.983333,...,1442.0,13.869626,Perimeter (<40%),40.0,2.545455,-111.818182,118.818182,Golden State Warriors,True,False
1,1630578,Alperen Sengun,Alperen Sengun,1610612745,Houston Rockets,22500001,2025-10-21,HOU @ OKC,L,48.890000,...,997.0,39.117352,Perimeter (<40%),40.0,6.454545,-109.909091,206.181818,Oklahoma City Thunder,False,False
2,1628983,Shai Gilgeous-Alexander,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,22500001,2025-10-21,OKC vs. HOU,W,47.216667,...,1558.0,25.802311,Perimeter (<40%),40.0,-6.454545,-109.909091,-251.727273,Houston Rockets,True,True


In [3]:
df = df_raw.copy()

num_cols = ["points_line", "points_over_odds", "points_under_odds", "PTS", "MIN",
            "team_spread", "FGM", "FGA", "FG3M", "FG3A", "FTM", "FTA",
            "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV",
            "pts_0_6_pct", "rim_fg_pct", "num_bookmakers"]
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["points_line", "PTS"]).copy()

required = ["PLAYER_NAME", "game_date", "MIN", "PTS", "points_line", "team_spread"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print(f"rows after dropna: {len(df):,}")
print(f"players: {df['PLAYER_NAME'].nunique():,}")
print(f"date range: {df['game_date'].min()} → {df['game_date'].max()}")
df.head(3)

rows after dropna: 11,062
players: 369
date range: 2025-10-21 → 2026-02-23


,PLAYER_ID,PLAYER_NAME,PLAYER_NAME_NORMALIZED,TEAM_ID,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,...,total_pts_season,pts_0_6_pct,scorer_type,rim_scorer_threshold,team_spread,team_spread_odds,team_moneyline,opponent_team,is_home,is_favorite
0,1629029,Luka Dončić,Luka Doncic,1610612747,Los Angeles Lakers,22500002,2025-10-21,LAL vs. GSW,L,40.983333,...,1442.0,13.869626,Perimeter (<40%),40.0,2.545455,-111.818182,118.818182,Golden State Warriors,True,False
1,1630578,Alperen Sengun,Alperen Sengun,1610612745,Houston Rockets,22500001,2025-10-21,HOU @ OKC,L,48.890000,...,997.0,39.117352,Perimeter (<40%),40.0,6.454545,-109.909091,206.181818,Oklahoma City Thunder,False,False
2,1628983,Shai Gilgeous-Alexander,Shai Gilgeous-Alexander,1610612760,Oklahoma City Thunder,22500001,2025-10-21,OKC vs. HOU,W,47.216667,...,1558.0,25.802311,Perimeter (<40%),40.0,-6.454545,-109.909091,-251.727273,Houston Rockets,True,True


---
## 2. EDA

Hypotheses going in:
1. Books are reasonably calibrated on points overall but shade certain player tiers.
2. Spread context (blowout risk) is a systematic pricing variable.
3. Scorer type (rim vs perimeter) adds predictive signal beyond the line itself.

In [4]:
# Target distribution
print("=== PTS distribution ===")
print(df["PTS"].describe().round(2))

print("\n=== Points line distribution ===")
print(df["points_line"].describe().round(2))

# Over/under/push rates
df["outcome"] = np.where(df["PTS"] > df["points_line"], "over",
                np.where(df["PTS"] < df["points_line"], "under", "push"))
print("\n=== Outcome rates ===")
print(df["outcome"].value_counts(normalize=True).round(3))

=== PTS distribution ===
count    11062.00
mean        14.18
std          8.80
min          0.00
25%          8.00
50%         13.00
75%         19.75
max         56.00
Name: PTS, dtype: float64

=== Points line distribution ===
count    11062.00
mean        14.04
std          6.34
min          2.50
25%          9.05
50%         12.75
75%         17.81
max         36.69
Name: points_line, dtype: float64

=== Outcome rates ===
outcome
under    0.515
over     0.483
push     0.002
Name: proportion, dtype: float64


In [5]:
# Market calibration — implied prob vs observed hit rate

def american_to_implied_prob(odds: float) -> float:
    if pd.isna(odds):
        return float("nan")
    return (-odds) / ((-odds) + 100.0) if odds < 0 else 100.0 / (odds + 100.0)

work = df[df["outcome"] != "push"].copy()
work["p_over"] = work["points_over_odds"].apply(american_to_implied_prob)
work["p_under"] = work["points_under_odds"].apply(american_to_implied_prob)
work["y_over"] = (work["outcome"] == "over").astype(int)
work["y_under"] = (work["outcome"] == "under").astype(int)

# Brier score (overall — aggregated odds, not per-book)
brier_over = ((work["p_over"] - work["y_over"]) ** 2).mean()
brier_under = ((work["p_under"] - work["y_under"]) ** 2).mean()
avg_implied_over = work["p_over"].mean()
observed_over = work["y_over"].mean()

print(f"Brier score (over side):  {brier_over:.4f}")
print(f"Brier score (under side): {brier_under:.4f}")
print(f"Avg implied over prob: {avg_implied_over:.3f}  |  Observed over rate: {observed_over:.3f}")
print(f"Calibration gap (over): {abs(avg_implied_over - observed_over):.3f}")

Brier score (over side):  0.2550
Brier score (under side): 0.2527
Avg implied over prob: 0.524  |  Observed over rate: 0.484
Calibration gap (over): 0.040


In [6]:
# Over rate by points_line tier — does the market shade certain tiers?
df["line_tier"] = pd.cut(
    df["points_line"],
    bins=[0, 10, 15, 20, 25, 30, 100],
    labels=["<10", "10-15", "15-20", "20-25", "25-30", "30+"]
)

tier_stats = (
    df[df["outcome"] != "push"]
    .groupby("line_tier", observed=True)
    .agg(
        n=("PTS", "count"),
        over_rate=("y_over", "mean") if "y_over" in df.columns else ("outcome", lambda x: (x == "over").mean()),
        avg_line=("points_line", "mean"),
        avg_pts=("PTS", "mean"),
    )
    .reset_index()
)

# recompute over_rate cleanly
no_push = df[df["outcome"] != "push"].copy()
no_push["y_over"] = (no_push["outcome"] == "over").astype(int)
tier_stats = (
    no_push.groupby("line_tier", observed=True)
    .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
         avg_line=("points_line", "mean"), avg_pts=("PTS", "mean"))
    .reset_index()
)
tier_stats.round(3)

,line_tier,n,over_rate,avg_line,avg_pts
0,<10,3493,0.486,7.565,7.961
1,10-15,3363,0.494,12.408,12.748
2,15-20,2228,0.474,17.315,17.266
3,20-25,1161,0.471,22.302,21.971
4,25-30,635,0.485,27.245,26.992
5,30+,159,0.440,31.883,30.403


In [7]:
# Over rate by spread bin — blowout risk effect
df["spread_bin"] = pd.cut(
    df["team_spread"],
    bins=[-30, -10, -5, 0, 5, 10, 30],
    labels=["big dog (>10)", "dog (5-10)", "slight dog (0-5)",
            "slight fav (0-5)", "fav (5-10)", "big fav (>10)"]
)
no_push["spread_bin"] = df.loc[no_push.index, "spread_bin"]

spread_stats = (
    no_push.groupby("spread_bin", observed=True)
    .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
         avg_pts=("PTS", "mean"), avg_line=("points_line", "mean"))
    .reset_index()
)
spread_stats.round(3)

,spread_bin,n,over_rate,avg_pts,avg_line
0,big dog (>10),1210,0.474,14.799,14.859
1,dog (5-10),2134,0.491,14.534,14.455
2,slight dog (0-5),2373,0.481,14.582,14.315
3,slight fav (0-5),2347,0.476,14.170,14.042
4,fav (5-10),1932,0.500,13.751,13.403
5,big fav (>10),1043,0.475,12.733,12.798


### VERDICT — EDA

*(fill in after running)*

- Calibration (Brier / calibration gap): ...
- Line tier shading (any tier consistently over/under 50%?): ...
- Spread effect (does blowout risk show up in the data?): ...
- Anything surprising: ...

---
## 3. Feature Ideation

Key candidate features — test each univariately vs `PTS` and vs `over_hit`:

| Feature | Hypothesis |
|---|---|
| `MIN` rolling avg | Minutes drive opportunity volume |
| `PTS` rolling avg (3/5/10g) | Recent scoring form |
| `team_spread` | Blowout risk / game pace effect |
| `points_line` | Proxy for player role/usage |
| `pts_0_6_pct` | Rim scoring pct — floor consistency signal |
| `scorer_type` | Rim vs perimeter |
| `is_home` | Home court scoring boost |
| `FTA` rolling avg | Free throw rate — trip to line floor |

In [8]:
# Build rolling features per player (shift(1) = no lookahead)
df_feat = df.sort_values(["PLAYER_NAME", "game_date"]).copy()

for window in [3, 5, 10]:
    df_feat[f"pts_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["PTS"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )
    df_feat[f"min_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["MIN"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )
    df_feat[f"fta_roll{window}"] = (
        df_feat.groupby("PLAYER_NAME")["FTA"]
        .transform(lambda s: s.shift(1).rolling(window, min_periods=window // 2).mean())
    )

df_feat["y_over"] = (df_feat["outcome"] == "over").astype(int)
df_feat["pts_diff"] = df_feat["PTS"] - df_feat["points_line"]
df_feat["is_home_num"] = df_feat["is_home"].astype(float) if "is_home" in df_feat.columns else np.nan

print(f"rows with pts_roll5: {df_feat['pts_roll5'].notna().sum():,}")
df_feat[["PLAYER_NAME", "game_date", "PTS", "points_line", "y_over",
          "pts_roll5", "min_roll5", "team_spread"]].head(5)

rows with pts_roll5: 10,338


,PLAYER_NAME,game_date,PTS,points_line,y_over,pts_roll5,min_roll5,team_spread
189,AJ Green,2025-10-22,11,8.666667,1,NaN,NaN,-9.863636
518,AJ Green,2025-10-24,9,8.285714,1,NaN,NaN,1.727273
800,AJ Green,2025-10-26,20,8.730769,1,10.000000,26.383333,6.500000
1264,AJ Green,2025-10-28,9,9.500000,0,13.333333,27.905556,2.181818
1575,AJ Green,2025-10-30,10,8.807692,1,12.250000,26.837500,7.681818


In [9]:
# Univariate correlations with PTS, y_over, pts_diff
feature_cols = (
    [f"pts_roll{w}" for w in [3, 5, 10]] +
    [f"min_roll{w}" for w in [3, 5, 10]] +
    [f"fta_roll{w}" for w in [3, 5, 10]] +
    ["team_spread", "points_line", "pts_0_6_pct", "is_home_num"]
)
feature_cols = [c for c in feature_cols if c in df_feat.columns]
targets = ["PTS", "y_over", "pts_diff"]

corr = df_feat[feature_cols + targets].corr()[targets].loc[feature_cols]
corr.round(3)

,PTS,y_over,pts_diff
pts_roll3,0.597,-0.009,-0.045
pts_roll5,0.627,-0.008,-0.044
pts_roll10,0.647,-0.008,-0.044
min_roll3,0.483,-0.002,-0.032
min_roll5,0.501,0.000,-0.029
min_roll10,0.510,-0.004,-0.033
fta_roll3,0.496,-0.001,-0.030
fta_roll5,0.532,-0.002,-0.030
fta_roll10,0.555,-0.004,-0.036
team_spread,-0.061,0.002,0.006


In [10]:
# Scorer type breakdown — does rim_scorer flag matter?
if "scorer_type" in df_feat.columns:
    scorer_stats = (
        df_feat[df_feat["outcome"] != "push"]
        .groupby("scorer_type")
        .agg(n=("PTS", "count"), over_rate=("y_over", "mean"),
             avg_pts=("PTS", "mean"), avg_line=("points_line", "mean"),
             avg_pts_0_6_pct=("pts_0_6_pct", "mean"))
        .reset_index()
    )
    print(scorer_stats.round(3))

           scorer_type     n  over_rate  avg_pts  avg_line  avg_pts_0_6_pct
0     Perimeter (<40%)  8515      0.491   14.948    14.734           23.201
1  Rim Attacker (≥40%)  2230      0.458   11.546    11.603           52.420


### VERDICT — Features

*(fill in after running)*

- Strongest predictors of PTS: ...
- Strongest predictors of y_over: ...
- Scorer type signal: ...
- Features to carry into baseline model: ...
- Features to drop: ...

---
## 4. Baseline Model

Goal: beat the consensus line RMSE with a simple model.
Naive baseline: predict = line. Compare OLS, Ridge, GBM.

In [11]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings("ignore")

FEATURE_COLS_CANDIDATES = [
    "pts_roll5", "min_roll5", "fta_roll5", "team_spread", "points_line",
    "pts_0_6_pct", "is_home_num",
]
FEATURE_COLS = [c for c in FEATURE_COLS_CANDIDATES if c in df_feat.columns]

# Core features required non-NaN; optional features filled with median
CORE_COLS = [c for c in ["pts_roll5", "min_roll5", "team_spread", "points_line"] if c in df_feat.columns]
model_df = df_feat.dropna(subset=CORE_COLS + ["PTS"]).copy()
for col in FEATURE_COLS:
    if col not in CORE_COLS and col in model_df.columns:
        median_val = model_df[col].median()
        model_df[col] = model_df[col].fillna(median_val if pd.notna(median_val) else 0)

# Final safety: drop any rows still NaN in feature matrix
model_df = model_df.dropna(subset=FEATURE_COLS + ["PTS"])

print("Null counts per feature after imputation:")
print(model_df[FEATURE_COLS].isnull().sum())
print(f"Model rows: {len(model_df):,}  features: {FEATURE_COLS}")

X = model_df[FEATURE_COLS].values
y = model_df["PTS"].values

# Naive baseline: predict = line
rmse_naive = np.sqrt(mean_squared_error(y, model_df["points_line"].values))

ols   = LinearRegression()
ridge = Ridge(alpha=1.0)
gbm   = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)

rmse_ols   = np.sqrt(-cross_val_score(ols,   X, y, cv=5, scoring="neg_mean_squared_error").mean())
rmse_ridge = np.sqrt(-cross_val_score(ridge, X, y, cv=5, scoring="neg_mean_squared_error").mean())
rmse_gbm   = np.sqrt(-cross_val_score(gbm,   X, y, cv=5, scoring="neg_mean_squared_error").mean())

results = pd.DataFrame({
    "model":   ["naive (line)", "OLS", "Ridge", "GBM"],
    "rmse_cv": [rmse_naive, rmse_ols, rmse_ridge, rmse_gbm],
})
results["improvement_vs_naive"] = (rmse_naive - results["rmse_cv"]) / rmse_naive
results.round(4)

Null counts per feature after imputation:
pts_roll5      0
min_roll5      0
fta_roll5      0
team_spread    0
points_line    0
pts_0_6_pct    0
is_home_num    0
dtype: int64
Model rows: 10,338  features: ['pts_roll5', 'min_roll5', 'fta_roll5', 'team_spread', 'points_line', 'pts_0_6_pct', 'is_home_num']


,model,rmse_cv,improvement_vs_naive
0,naive (line),6.4308,0.0000
1,OLS,6.4221,0.0014
2,Ridge,6.4221,0.0014
3,GBM,6.4510,-0.0031


In [12]:
# GBM feature importances
gbm_fit = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
gbm_fit.fit(X, y)

pd.Series(gbm_fit.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False).round(4)

points_line    0.9688
pts_roll5      0.0097
fta_roll5      0.0066
pts_0_6_pct    0.0055
min_roll5      0.0052
team_spread    0.0040
is_home_num    0.0002
dtype: float64

### VERDICT — Baseline Model

*(fill in after running)*

- Best model: ...
- RMSE improvement vs naive: ...
- Top features by importance: ...
- Proceed to edge calibration: yes / no

---
## 5. Edge Calibration

Given model prediction vs line, when does a bet have positive EV?
Parametric normal around model mean (same approach as rebounds Option A).

In [13]:
from scipy.stats import norm

model_df["pred"] = gbm_fit.predict(X)
model_df["resid"] = model_df["PTS"] - model_df["pred"]

sigma = model_df["resid"].std()
print(f"Residual sigma: {sigma:.3f} pts")
print(model_df["resid"].describe().round(3))

Residual sigma: 6.321 pts
count    10338.000
mean        -0.000
std          6.321
min        -24.277
25%         -4.215
50%         -0.426
75%          3.807
max         31.414
Name: resid, dtype: float64


In [14]:
# P(PTS > line) under model distribution
model_df["p_over_model"] = norm.sf(model_df["points_line"], loc=model_df["pred"], scale=sigma)
model_df["p_over_market"] = model_df["points_over_odds"].apply(american_to_implied_prob)
model_df["edge_over"] = model_df["p_over_model"] - model_df["p_over_market"]

model_df["y_over"] = (model_df["outcome"] == "over").astype(int)

model_df["edge_bin"] = pd.cut(
    model_df["edge_over"],
    bins=[-1, -0.10, -0.05, 0.0, 0.05, 0.10, 1.0],
    labels=["<-10%", "-10 to -5%", "-5 to 0%", "0 to 5%", "5 to 10%", ">10%"]
)

edge_summary = (
    model_df.dropna(subset=["edge_bin", "p_over_market", "y_over"])
    .groupby("edge_bin", observed=True)
    .agg(
        n=("y_over", "count"),
        over_rate=("y_over", "mean"),
        avg_edge=("edge_over", "mean"),
        avg_p_market=("p_over_market", "mean"),
    )
)
edge_summary["implied_roi"] = (edge_summary["over_rate"] / edge_summary["avg_p_market"]) - 1
edge_summary.round(3)

,n,over_rate,avg_edge,avg_p_market,implied_roi
edge_bin,,,,,
<-10%,595,0.319,-0.149,0.544,-0.413
-10 to -5%,1953,0.436,-0.070,0.537,-0.188
-5 to 0%,4253,0.489,-0.023,0.535,-0.087
0 to 5%,3083,0.522,0.019,0.524,-0.005
5 to 10%,265,0.566,0.063,0.509,0.112
>10%,189,0.524,0.434,0.099,4.304


### VERDICT — Edge Calibration

*(fill in after running)*

- Edge threshold where ROI turns positive: ...
- Sample size at that threshold: ...
- Over vs under asymmetry (under bias like rebounds?): ...
- Next step: OOS backtest on prior seasons

---
## Scratch

Ad-hoc exploration. Code here does NOT graduate to scripts.